<a href="https://colab.research.google.com/github/p-tech/BDAI-Files/blob/main/log_files/log_file_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab: Real-World Web Log Analysis
In this session, we will analyze traffic from www.almos.org.uk. This log contains a mix of real users, search engine crawlers, and automated system scripts.

Load LogFile

In [ ]:
from google.colab import files
import pandas as pd
import io

# This will prompt the student to select the file from their computer
uploaded = files.upload()

# Get the filename (assuming only one file is uploaded)
filename = list(uploaded.keys())
print(f"Successfully uploaded: {filename}")

Import log file into Pandas

### Parsing the Logs
We are using a Regular Expression (Regex) to extract 10 specific pieces of information from each line:
1. **Domain** (e.g., www.almos.org.uk)
2. **IP** (Supporting both IPv4 and IPv6)
3. **Identity/Auth** (Usually `-`)
4. **Timestamp** (Inside the `[]`)
5. **Request** (The method and URL)
6. **Status** (The HTTP response code)
7. **Size** (Bytes sent)
8. **Referer** (Where the user came from)
9. **User Agent** (The browser or bot name)

In [ ]:
import pandas as pd
import re

# This pattern matches your specific log format exactly
log_pattern = r'^(?P<domain>\S+)\s(?P<ip>\S+)\s(?P<ident>\S+)\s(?P<auth>\S+)\s\[(?P<timestamp>.*?)\]\s"(?P<request>.*?)"\s(?P<status>\d+)\s(?P<size>\S+)\s"(?P<referer>.*?)"\s"(?P<user_agent>.*?)"'

# Replace 'your_uploaded_file.log' with the actual name of your file
file_path = 'access_log'

data = []
with open(file_path, 'r') as f:
    for line in f:
        match = re.search(log_pattern, line)
        if match:
            data.append(match.groupdict())

# Create the DataFrame
df = pd.DataFrame(data)

print(f"Successfully loaded {len(df)} lines into the DataFrame.")
df.head()

Look at the output. We have mixed IPv4 and IPv6 addresses, status codes like 429 (Too Many Requests), and clear "User-Agent" strings identifying bots (Facebook, Google, and WordPress).

**Step 1:** Identifying Bot Traffic
In this log, the Facebook crawler (meta-externalagent) is being Rate Limited.

Status Code 429: Means "Too Many Requests." The server is telling the bot to slow down.

**Step 2:** Spotting WordPress Automation
Look for wp-cron.php. This is an internal WordPress task manager, not a real user.

**Cleaning the Data Types**
By default, everything we just pulled is a "string" (text). To do math or sort by date, we need to convert them to numbers and real timestamps.

In [ ]:
# 1. Convert Status to integers (for numerical analysis)
df['status'] = df['status'].astype(int)

# 2. Convert Size to numbers (some lines have '-' which we turn into 0)
df['size'] = pd.to_numeric(df['size'], errors='coerce').fillna(0).astype(int)

# 3. Convert Timestamp to a real Python datetime object
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d/%b/%Y:%H:%M:%S %z')

print("Data types converted successfully!")
df.info()

# Code Exploration

In [ ]:
print("--- 1. Breakdown of HTTP Status Codes ---")
# Students should see 429, 200, 301, and 302 here
print(df['status'].value_counts())

print("\n--- 2. Top User Agents (Who is visiting?) ---")
# This helps identify Googlebot, Facebook, and WordPress
print(df['user_agent'].value_counts())

# Filter for 429s
rate_limited = df[df['status'] == 429]

if rate_limited.empty:
    print("❌ No 429 errors found. Check if the Regex captured the 'status' column correctly.")
    print("Current column names:", df.columns.tolist())
else:
    print(f"✅ Found {len(rate_limited)} rate-limited requests.")
    # Use .head() to avoid flooding the screen
    print(rate_limited[['ip', 'status', 'request']].head(10))

In [ ]:
# Force conversion to numeric, turning errors into NaN
df['status'] = pd.to_numeric(df['status'], errors='coerce')

# Filter for 429s
rate_limited = df[df['status'] == 429]

if rate_limited.empty:
    print("❌ No 429 errors found. Check if the Regex captured the 'status' column correctly.")
    print("Current column names:", df.columns.tolist())
else:
    print(f"✅ Found {len(rate_limited)} rate-limited requests.")
    # Use .head() to avoid flooding the screen
    print(rate_limited[['ip', 'status', 'request']].head(10))

# Identifying and Categorizing Bots
In this cell, we will create a new column called is_bot and then filter for the most common ones.

In [ ]:
# Define common bot keywords
bot_keywords = 'bot|crawler|spider|slurp|externalagent|wordpress|serf'

# Create a boolean column for bots (case-insensitive)
df['is_bot'] = df['user_agent'].str.contains(bot_keywords, case=False, na=False)

# Create a separate DataFrame for just the bots
bots_df = df[df['is_bot'] == True].copy()

print(f"Detected {len(bots_df)} bot-related requests out of {len(df)} total lines.")

# Listing the "Main" Bots
To make this readable for students, we should group the bots by their User-Agent and count them.

In [ ]:
print("--- Top 10 Bots by Request Volume ---")
# We count how many times each unique User-Agent appears
top_bots = bots_df['user_agent'].value_counts().head(10)

# Display as a clean table
print(top_bots.to_frame(name='Request Count'))

# Optional: Find the most active Bot IP
print("\n--- Most Active Bot IP Addresses ---")
print(bots_df['ip'].value_counts().head(5))

In [ ]:
import matplotlib.pyplot as plt

# Count Bot vs Human
traffic_type = df['is_bot'].value_counts()
traffic_type.index = ['Bot', 'Potential Human'] # Rename for the chart

# Plot
plt.figure(figsize=(8, 6))
traffic_type.plot(kind='pie', autopct='%1.1f%%', colors=['#ff9999','#66b3ff'], startangle=90)
plt.title('Traffic Composition: Bots vs. Others')
plt.ylabel('') # Remove the default 'is_bot' label
plt.show()

# Exporting Bot Data for Further Investigation
This step is useful for sharing findings with a security team or keeping a record of specific bot behaviors.

In [ ]:
# 1. Filter for bots AND exclude the localhost IP (127.0.0.1)
# We use the != operator to "keep everything that is NOT 127.0.0.1"
external_bots_df = df[(df['is_bot'] == True) & (df['ip'] != '127.0.0.1')].copy()

# 2. Sort by IP address and then by Timestamp
bots_sorted = external_bots_df.sort_values(by=['ip', 'timestamp'])

# 3. Export to CSV
output_filename = 'filtered_bots_analysis.csv'
bots_sorted.to_csv(output_filename, index=False)

print(f"Total lines in original data: {len(df)}")
print(f"Lines excluded (Localhost): {len(df[df['ip'] == '127.0.0.1'])}")
print(f"Final bot records exported: {len(bots_sorted)}")

# Trigger the download in Colab
from google.colab import files
files.download(output_filename)

# **Visualizing Traffic Volume with a Treemap**
This visualization helps students immediately see the "Heavy Hitters"—IPs that might be scraping large amounts of data.

In [ ]:
import plotly.express as px

# 1. Prepare data: Filter for bots, exclude localhost
bots_only = df[(df['is_bot'] == True) & (df['ip'] != '127.0.0.1')].copy()

# 2. Group by both IP and User-Agent to see the breakdown
bot_summary = bots_only.groupby(['ip', 'user_agent']).agg(
    access_count=('ip', 'count'),
    total_bytes=('size', 'sum')
).reset_index()

# 3. Draw the Treemap
# 'path' defines the hierarchy: IP -> User Agent
fig = px.treemap(
    bot_summary,
    path=[px.Constant("All Bots"), 'ip', 'user_agent'],
    values='access_count',
    color='total_bytes',
    color_continuous_scale='RdBu',
    title='Bot Fingerprinting: IP Address & User-Agent Breakdown',
    hover_data=['total_bytes']
)

fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig.show()

Shared Identities: They might see one IP box contain multiple smaller boxes (User-Agents). This often indicates a proxy or a single server running multiple different crawling scripts.

Volume vs. Identity: A bot might have a very "loud" identity (many hits) but very "light" data footprint (darker/different color).

Bot Fingerprinting: If students see the same User-Agent appearing under ten different IP addresses, they’ve just identified a distributed botnet or a large-scale crawler like Facebook's meta-externalagent.

# Mapping the Bot's Path
We will create a function that allows a student to input an IP address and get a "Click Stream" of that bot's activity.

In [ ]:
# Choose a specific IP from your Treemap (e.g., the Facebook crawler)
target_ip = '2a03:2880:f804:3::'

# Filter for just this IP
bot_activity = df[df['ip'] == target_ip].copy()

# Group by the 'request' column to see which URLs were targeted
url_counts = bot_activity['request'].value_counts().reset_index()
url_counts.columns = ['URL Requested', 'Hit Count']

print(f"--- Activity Report for IP: {target_ip} ---")
print(f"Total Requests: {len(bot_activity)}")
print(f"Unique URLs Accessed: {len(url_counts)}")
print("\nTop Requested URLs:")
print(url_counts.head(30))

Identifying Intent
Have the students analyze the URL list for their chosen IP:

The "Good" Bot: Does it mostly access public articles and CSS files? (e.g., Googlebot indexing content).

The "Aggressive" Bot: Does it have a high number of 429 status codes? (This means the server is actively blocking it for being too fast).

The "System" Bot: Does it only hit wp-cron.php? (This is your internal WordPress system keeping the site running).

In [ ]:
import plotly.express as px

# Create a scatter plot of requests over time for this specific bot
fig = px.scatter(
    bot_activity,
    x='timestamp',
    y='request',
    color='status',
    title=f'Timeline of Requests from {target_ip}',
    labels={'timestamp': 'Time of Day', 'request': 'URL Path'},
    hover_data=['status', 'size']
)

fig.update_layout(height=600)
fig.show()

The "Wall of Shame" (Suspicious Bot Detection)We will rank bots based on their Error Ratio. If an IP has a high percentage of $403$ (Forbidden), $404$ (Not Found), or $429$ (Rate Limited) responses, it is likely a malicious scanner or an aggressive scraper.

We will now calculate a **Suspicion Score**.
* **High Hits + High Errors ($4xx$):** This usually indicates a "Scraper" or "Vulnerability Scanner."
* **Targeting Admin Paths:** Any IP hitting `/wp-admin` or `.env` files is flagged immediately.



In [ ]:
# 1. Define the 'Threat Dictionary'
status_labels = {
    401: "Unauthorized (Login Probe)",
    403: "Forbidden (Access Denied)",
    404: "Not Found (Directory Scanning)",
    405: "Method Not Allowed (Exploit Attempt)",
    429: "Rate Limited (Aggressive Scraper)"
}

# 2. Add a 'threat_type' column to the main DataFrame
df['threat_type'] = df['status'].map(status_labels).fillna("Normal / Other")

# 3. Create a summary of threats per IP
threat_summary = df[df['status'].isin(status_labels.keys())].groupby(['ip', 'threat_type']).size().reset_index(name='incident_count')

# 4. Sort to see the most active "Threats"
threat_summary = threat_summary.sort_values(by='incident_count', ascending=False)

print("--- Categorized Security Incidents ---")
display(threat_summary.head(25))

In your specific log file, you have several 429 and 301/302 codes. Adding this section helps students distinguish between a "Bot doing its job" and a "Bot being bad."

404 (Not Found): If an IP hits 50 different 404s in 1 minute, they are "Fuzzing"—guessing URL names to find a forgotten backup or a hidden login page.

405 (Method Not Allowed): This often happens if a hacker tries to POST data (like a password) to a page that only allows GET (viewing). It's a huge red flag.

401 (Unauthorized): In a WordPress context, this usually means someone tried to access /wp-admin without a session cookie or with a failed login attempt.

In [ ]:
import plotly.express as px

# Only plot IPs that triggered at least one of our defined security codes
fig = px.bar(
    threat_summary,
    x='ip',
    y='incident_count',
    color='threat_type',
    title='Security Incident Breakdown by IP Address',
    labels={'incident_count': 'Number of Errors', 'ip': 'Attacker IP'},
    barmode='stack'
)

fig.show()

# Analyzing Real Human Traffic
Now we are filtering the data to see only what **real people** (non-bots) are reading.
We will exclude:
* **Bots** (as identified by User-Agent)
* **Assets** (Images, CSS, JavaScript, Fonts)
* **Internal System Calls** (Localhost 127.0.0.1)

In [ ]:
# 1. Define the "Noise" extensions to exclude
noise_extensions = r'\.jpg|\.jpeg|\.png|\.gif|\.css|\.js|\.woff|\.svg|\.ico'

# 2. Apply the filters
# - Must NOT be a bot
# - Must NOT be 127.0.0.1
# - Must NOT contain noise extensions in the request URL
human_traffic = df[
    (df['is_bot'] == False) &
    (df['ip'] != '127.0.0.1') &
    (~df['request'].str.contains(noise_extensions, case=False, na=False))
].copy()

# 3. Clean up the 'request' string to just show the URL (remove 'GET ' and ' HTTP/1.1')
human_traffic['clean_url'] = human_traffic['request'].str.replace(r'^GET |^POST | HTTP/1.1$', '', regex=True)

print(f"Total Log Entries: {len(df)}")
print(f"Actual Content Pages viewed by humans: {len(human_traffic)}")

# Display the top 10 most popular pages
print("\n--- Most Popular Content Pages (Human Only) ---")
top_pages = human_traffic['clean_url'].value_counts().head(10)
display(top_pages.to_frame(name='Views'))

# The "Reading List" Report
This helps students see exactly what the audience is interested in.

In [ ]:
# Group by URL and show the Status Codes (to see if humans are hitting errors)
content_report = human_traffic.groupby(['clean_url', 'status']).size().reset_index(name='view_count')
content_report = content_report.sort_values(by='view_count', ascending=False)

# Export this for your records
content_report.to_csv('human_content_views.csv', index=False)

print("Report saved as 'human_content_views.csv'")
display(content_report.head(15))

In [ ]:
import plotly.express as px

# Count views per hour/minute
human_traffic.set_index('timestamp', inplace=True)
time_series = human_traffic.resample('1min').size().reset_index(name='page_views')

fig = px.line(time_series, x='timestamp', y='page_views', title='Human Page Views Over Time')
fig.show()

# Reset index for further cells
human_traffic.reset_index(inplace=True)

### 📈 Step 14: Unique Visitor Analysis
In this step, we identify individual "journeys."
We are looking for:
1. **Who?** (Unique IP Address)
2. **When?** (The specific Date of access)
3. **How much?** (Count of unique URLs visited)

In [ ]:
# 1. Create a 'Date' only column (ignoring the specific hour/minute)
human_traffic['entry_date'] = human_traffic['timestamp'].dt.date

# 2. Group by IP and Date
# We calculate:
# - total_pages: Total number of clicks
# - unique_pages: Number of different URLs they looked at
visitor_stats = human_traffic.groupby(['ip', 'entry_date']).agg(
    total_clicks=('clean_url', 'count'),
    unique_pages_viewed=('clean_url', 'nunique')
).reset_index()

# 3. Sort by the most active "Human" visitors
visitor_stats = visitor_stats.sort_values(by='unique_pages_viewed', ascending=False)

print(f"Detected {len(visitor_stats)} unique IP/Date sessions.")
display(visitor_stats.head(15))

# Visualizing Visitor Depth
It's helpful for students to see the distribution. Are most people looking at 1 page, or are they browsing 10+ pages?

In [ ]:
import plotly.express as px

fig = px.histogram(
    visitor_stats,
    x='unique_pages_viewed',
    nbins=20,
    title='Distribution of Visitor Depth (Pages per Session)',
    labels={'unique_pages_viewed': 'Number of Unique Pages Accessed', 'count': 'Number of Visitors'},
    color_discrete_sequence=['#2ecc71']
)

fig.update_layout(bargap=0.1)
fig.show()

In [ ]:
visitor_stats.to_csv('unique_human_visitors.csv', index=False)
print("Visitor Journey Report saved as 'unique_human_visitors.csv'")

# Reverse DNS Lookup
A **Reverse DNS (rDNS)** lookup asks the internet: "What name is registered to this IP address?"
This helps us verify:
* Is a bot truly from Google or Facebook?
* Is a visitor using a known VPN or ISP (like Comcast or British Telecom)?

In [ ]:
import socket

def get_domain(ip):
    try:
        # gethostbyaddr returns (hostname, aliaslist, ipaddrlist)
        # We only want the primary hostname
        return socket.gethostbyaddr(ip)
    except (socket.herror, socket.gaierror, Exception):
        # Return 'Unknown' if the lookup fails or takes too long
        return "Unknown / No PTR Record"

# 1. Get unique IPs from our visitor stats to save time
# (No need to lookup the same IP 100 times!)
unique_ips = visitor_stats['ip'].unique()

# 2. Create a lookup dictionary
print(f"Performing reverse lookup for {len(unique_ips)} unique IPs...")
ip_to_domain = {ip: get_domain(ip) for ip in unique_ips}

# 3. Map the domains back to our visitor table
visitor_stats['domain'] = visitor_stats['ip'].map(ip_to_domain)

display(visitor_stats[['ip', 'domain', 'unique_pages_viewed']].head(15))

# Identifying "Fakers"
Students can now write a simple check: "If the User-Agent says 'Googlebot' but the Domain does not contain 'googlebot.com', is it a fake?"

In [ ]:
# Quick check for suspicious authenticity
# (Assuming you still have the User-Agent data in your main 'df')
verification = df[['ip', 'user_agent']].drop_duplicates()

# Map the resolved domains
verification['resolved_domain'] = verification['ip'].map(ip_to_domain)

# Ensure the 'resolved_domain' column contains only strings (the hostname)
# If the entry is a tuple, extract the first element (hostname); otherwise, use the entry as is.
verification['resolved_domain'] = verification['resolved_domain'].apply(
    lambda x: x[0] if isinstance(x, tuple) else x
)

# Look for 'Google' in Agent but NOT in Domain
fakes = verification[
    (verification['user_agent'].str.contains('Google', case=False, na=False)) &
    (~verification['resolved_domain'].str.contains('google', case=False, na=False))
]

if not fakes.empty:
    print("🚨 Potential FAKE Googlebots detected:")
    display(fakes)
else:
    print("✅ No obvious bot spoofing detected in this sample.")

# Final Export of Human Visitor Data
We are now exporting our analyzed "Human" traffic. This file contains:
1. **IP Address** 2. **Entry Date**
3. **Domain Name** (via Reverse DNS)
4. **Volume Metrics** (Total clicks vs. Unique pages)

In [ ]:
# 1. Ensure the 'domain' column is mapped to our visitor_stats
visitor_stats['domain'] = visitor_stats['ip'].map(ip_to_domain)

# 2. Final Sorting (Most active visitors first)
final_export_df = visitor_stats.sort_values(by=['unique_pages_viewed', 'entry_date'], ascending=False)

# 3. Define the filename
csv_filename = 'human_visitor_audit_2025.csv'

# 4. Save to the Colab environment
final_export_df.to_csv(csv_filename, index=False)

print(f"✅ Success! Generated {csv_filename} with {len(final_export_df)} unique sessions.")

# 5. Trigger the browser download
from google.colab import files
files.download(csv_filename)

# Deep Dive into a Single Visitor's Path
This cell allows you (or your students) to pick one interesting IP address and see their entire chronological "clickstream."

In [ ]:
# 1. Pick an IP address to investigate (replace with one from your visitor_stats)
target_ip = '66.249.64.131'  # Example: A Googlebot IP or a specific human IP

# 2. Filter the human_traffic for just this IP
# We sort by timestamp so we see the pages in the order they were clicked
visitor_journey = human_traffic[human_traffic['ip'] == target_ip].sort_values(by='timestamp')

# 3. Display the results
print(f"--- Full Clickstream for IP: {target_ip} ---")
print(f"Total Pages Viewed: {len(visitor_journey)}")

# Show the specific columns: Time, the URL, and the Status Code
display(visitor_journey[['timestamp', 'clean_url', 'status']])